# Topic: Secure Serialization - Pickle deserialization vulnerabilities, __reduce__ exploit mechanisms, and secure JSON formats
 
## 1. THE SECURITY RISKS OF PICKLE
- **`pickle`**: Python's native object serialization module. Converts a Python object hierarchy into a byte stream (pickling) and back (unpickling).
- **The Security Hazard**: Python's unpickling process evaluates a mini execution stack machine. If a serialized class payload contains a custom **`__reduce__(self)`** method, it instructs `pickle` to call an arbitrary function (like `os.system`) with arbitrary arguments when unpickling.
- **Rule**: **NEVER unpickle untrusted data** received over the network or from user inputs. Doing so allows trivial Remote Code Execution (RCE).
 
## 2. SAFE ALTERNATIVES
- **JSON**: Use `json.dumps()` / `json.loads()`. JSON is a static, pure data-interchange format. It does not support class instantiation or runtime function execution, making it secure against code execution exploits.
- **Protobuf / MessagePack**: Binary formats that are pure schema-based data representation formats with no execution runtime engines.
 
---


In [ ]:
import pickle
import json
import os

# 1. Simulating an Exploit Payload using pickle.__reduce__
class MaliciousPayload:
    """Class designed to trigger command execution during unpickling."""
    def __reduce__(self):
        # Returns a tuple: (callable_to_execute, tuple_of_arguments)
        # os.system is a standard target for proof-of-concept exploits
        return (os.system, ("echo 'REMOTE CODE EXECUTED VIA PICKLE!'",))

print("--- Creating Exploit Bytes ---")
exploit_obj = MaliciousPayload()
# Serialize payload
serialized_exploit = pickle.dumps(exploit_obj)
print(f"Serialized Byte Stream: {serialized_exploit}")



In [ ]:
# 2. Simulating Vulnerable Deserialization
print("\n--- Deserializing Payload using pickle.loads() ---")
try:
    # Unpickling triggers __reduce__ immediately, executing os.system!
    pickle.loads(serialized_exploit)
except Exception as e:
    print(f"Error: {e}")
# Expected: Runs the system echo command directly!



In [ ]:
# 3. Secure Serialization alternative (JSON)
print("\n--- Using Secure JSON Format ---")
# JSON only allows primitives (dict, list, string, number, bool, null)
# It cannot execute code or initiate classes.
data_to_serialize = {
    "user_id": "dev_user",
    "roles": ["developer", "auditor"]
}

# Serialize to JSON
json_bytes = json.dumps(data_to_serialize).encode('utf-8')
print(f"JSON Byte Stream: {json_bytes}")

# Deserialize (completely safe!)
parsed_data = json.loads(json_bytes.decode('utf-8'))
print(f"Parsed JSON dictionary: {parsed_data}")

# Attempt to pass pickle exploit bytes to json.loads (will throw an error)
try:
    json.loads(serialized_exploit.decode('utf-8'))
except Exception as e:
    print(f"Caught expected JSON parsing exception: {e}")
